# Hidden Markov Model
Hidden Markov Models (HMMs) contain hidden states that we are trying to infer from observed data. This is useful in bioinformatics, because the observed data is what we can directly measure, like sequenced DNA. On the other hand, the hidden states represent the underlying biological context we are trying to uncover or infer. We will use the Viterbi algorithm to find the most likely sequence of hidden states given a sequence of observations.

The Baum-Welch algorithm represents an Expectation-Maximization approach to learning HMM parameters, offering several advantages over manual parameter setting:

Key Features

* Unsupervised Learning: Estimates model parameters without labeled training data
* Maximum Likelihood: Finds parameters that maximize observation probability
* Iterative Refinement: Progressively improves parameter estimates
* Convergence Guarantees: Always reaches a local optimum of the likelihood function

*
* Algorithm Structure

The Baum-Welch algorithm involves these key steps:

**Initialization:**
* Start with initial guesses for transition, emission, and initial probabilities
* Set up convergence criteria and pseudocounts to prevent zero probabilities

**Expectation Step (E):**
* Run Forward-Backward algorithm on training sequences
* Calculate expected counts for transitions and emissions
* Compute posterior probabilities for each state at each position
* These expected counts represent how often each transition and emission is used to generate the observed sequences

**Maximization Step (M):**
* Update model parameters based on expected counts
* Re-estimate initial, transition, and emission probabilities using the expected counts
* Normalize to ensure valid probability distributions
* Scale values to prevent numerical underflow

**Iteration and Convergence:**
* Repeat the E and M steps until convergence criteria are met
* Monitor likelihood improvement between iterations
* Handle multiple observation sequences appropriately


In [1]:
import numpy as np
from hmm_utils import ForwardBackward

In [8]:
class BaumWelch(ForwardBackward):
    def __init__(self):
        self.emission_counts = np.full((len(self.states), 4), -np.inf)
        self.transition_counts = np.full((len(self.states), len(self.states)), -np.inf)
        self.initial_counts = np.full((len(self.states),), -np.inf)
        self.total_seq_prob = -np.inf

    def initialize(self):
        pass

    def expectation(self, observations):
        """
        # obs = ["GGCACTGAA", "ATGCAATGC", "AATGCCTGA"]
        total_prob = 0
        for observation in observations:
            forward = Build forward_matrix(observation)
            backward = Build backward_matrix(observation)
            posterior = forward_backward(observations)
            xi = compute_transition_posterior(forward, backward, observation)
            accumulate emission counts (posterior, sequence)
            accumulate transition counts (xi, sequence)
            accumulate initial counts (posterior)
        """

        for obs in observations:
            # Create intermediate matrices
            forward, backward, posterior = self.forward_backward(obs)
            transition_posterior = self.transition_posterior_matrix(forward, backward, obs)

            # Accumulation
            self.accumulate_emission_counts(posterior)
            self.accumulate_transition_counts(transition_posterior)
            self.accumulate_initial_counts(posterior)
            self.accumulate_total_seq_prob(forward)


    def transition_posterior_matrix(self, forward, backward, observation):
        # Create a 3D matrix filled with -inf (log-space)
        trans_posterior = np.full((len(self.states),len(observation)-1, len(self.states)), -np.inf)
        log_trans_probs = np.log(self.transition_probs)
        log_emission_probs = np.log(self.emission_probs)
        for n in range(len(observation)-1):
            # Vectorization
           trans_posterior[:, n] = forward[:, n][:, np.newaxis] + log_trans_probs + log_emission_probs[self.nucleotide_map[observation[n+1]]] + backward[:, n+1][np.newaxis, :]
        return trans_posterior

    def accumulate_emission_counts(self, posterior, sequence):
        for i, pos in enumerate(sequence):
            pos_index = self.nucleotide_map[pos]

            self.emission_counts[:, pos_index] = np.logaddexp(self.emission_counts[:, pos_index], posterior[:, i])

    def accumulate_transition_counts(self, trans_posterior):
        self.transition_counts = np.logaddexp(self.transition_counts, trans_posterior)

    def accumulate_initial_counts(self, posterior):
       self.initial_counts = np.logaddexp(self.initial_counts, posterior[:, 0])

    def accumulate_total_seq_prob(self, forward):
        self.total_seq_prob = np.logaddexp(self.total_seq_prob, self.sequence_probability(forward))

    def normalization(self):
        # Normalize initial counts by total probability of accumulated sequences
        normalized_initial_counts = self.initial_counts - self.total_seq_prob

        # Normalize emission counts
        posterior_probs_sum_by_states = np.logaddexp.reduce(self.emission_counts, axis=1)
        normalized_emission_counts = self.emission_counts - posterior_probs_sum_by_states

        # Normalize transition counts
        trans_posterior_probs_sum_by_states = np.logaddexp.reduce(self.transition_counts, (axis=1,2))
        normalized_transition_counts = self.transition_counts - trans_posterior_probs_sum_by_states

        return normalized_initial_counts, normalized_emission_counts, normalized_transition_counts

    def maximization(self, normalized_initial_counts, normalized_emission_counts, normalized_transition_counts):
        # Issue do i need to unlog anything

        # Build the new emission probability
        self.emission_probs = normalized_emission_counts

        # Build the new transitions probability
        self.transition_probs = normalized_transition_counts

        # Build the new initial probability
        self.initial_probs = normalized_initial_counts


In [5]:
# Example observation sequences (multiple sequences for training)
obs = ["GGCACTGAA", "ATGCAATGC", "AATGCCTGA"]

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "H": 0.5,  # H = High GC content state
    "L": 0.5   # L = Low GC content state
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "H": {"H": 0.6, "L": 0.4},
    "L": {"H": 0.3, "L": 0.7}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "H": {"A": 0.2, "C": 0.3, "G": 0.3, "T": 0.2},
    "L": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

In [6]:
# Example observation sequence following the powerpoint
obs = "ACGCGATC"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.1,
    "G": 0.9
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.6, "G": 0.4},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.4, "C": 0.1, "G": 0.1, "T": 0.4}
}

hmm = ForwardBackward(init_probs, trans_probs, emit_probs)

print(f"Forward: {hmm.forward_matrix(obs)}\n\nBackward: {hmm.backward_matrix(obs)}")

print(f"\nPosterior Matrix: ")
fb_matrix = hmm.forward_backward(obs)
print(fb_matrix)

print(f"\nPosterior Decoded: {hmm.posterior_decode(obs)}")
viterbi = hmm.viterbi_algorithm(obs)
print(f"\nViterbi Matrix: {viterbi}")

index_to_check = 4
print(f"\nProbabilities of states at index: {index_to_check}")
posteriors_at_index = hmm.get_posterior_at_index(index_to_check)
for state, prob in zip(hmm.states, posteriors_at_index):
    print(f"{state} : {prob}")



Forward: [[ -4.60517019  -4.08637639  -5.23178084  -6.55181661  -7.917803
  -10.68397931 -12.96631674 -13.07609226]
 [ -1.02165125  -3.41732676  -5.62017689  -7.52408958  -9.15496622
   -9.24781431 -10.16898167 -12.55018913]]

Backward: [[ -9.82510559  -8.4447688   -7.07732394  -5.75233427  -4.58615218
   -3.28075123  -1.27296568   0.        ]
 [-11.16497558  -9.69183725  -8.07448111  -6.19061817  -4.00457699
   -3.00376445  -2.04022083   0.        ]]

Posterior Matrix: 
[[-2.34446282 -0.44533223 -0.22329182 -0.21833791 -0.41814222 -1.87891758
  -2.15346945 -0.9902793 ]
 [-0.10081387 -1.02335106 -1.60884504 -1.62889479 -1.07373024 -0.1657658
  -0.12338953 -0.46437617]]

Posterior Decoded: ['G', 'I', 'I', 'I', 'I', 'G', 'G', 'G']
['I', 'G']

Viterbi Matrix: [['G', 'I', 'I', 'I', 'I', 'G', 'G', 'G']]

Probabilities of states at index: 4
I : -0.41814221787486083
G : -1.0737302429154543
